# nn-module-subclass composite — cx19: subclass nn.Module and compose two registered submodules

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `nn-module-subclass`, `module-composition`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "nn-module-subclass"
DD_ATOM_IDS = ["nn-module-subclass", "module-composition"]
DD_SUBTOPICS = ["PyTorch: nn.Module subclassing", "PyTorch: Module composition"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

Building a CNN-shaped module is two atoms in lockstep:
- **nn-module-subclass** — write `class Foo(nn.Module):` with `__init__` calling `super().__init__()` and a `forward(self, x)` method. The base-class init is what wires up the bookkeeping dicts (`_parameters`, `_modules`, `_buffers`) PyTorch uses to walk the tree.
- **module-composition** — assign child modules as ATTRIBUTES inside `__init__`. The custom `nn.Module.__setattr__` watches for any value that is itself an `nn.Module` and quietly registers it in `self._modules` under the attribute name. That registration is what makes `.parameters()`, `.to(device)`, `.train()`, and `.state_dict()` recurse into the children.

**Anatomy.**
1. `super().__init__()` — MUST run before any `self.<child> = ...` assignment, otherwise `_modules` does not exist yet and the auto-registration silently breaks.
2. `self.fc1 = nn.Linear(...)` / `self.fc2 = nn.Linear(...)` — each assignment registers the child. The attribute name is the registration key.
3. `forward(self, x)` — call the children in order. `self.fc1(x)` invokes their `__call__`, which wraps `forward` with hooks and gradient bookkeeping.

### Composite Exercise — subclass nn.Module and compose two registered submodules

**Atoms exercised together**: `nn-module-subclass`, `module-composition`

Define a class `TwoLayer(nn.Module)` and a builder `cx19_build_two_layer(in_dim, hid_dim, out_dim)` that returns an instance.

`TwoLayer.__init__` must:
1. Call `super().__init__()` first.
2. Assign `self.fc1 = nn.Linear(in_dim, hid_dim)`.
3. Assign `self.fc2 = nn.Linear(hid_dim, out_dim)`.

`TwoLayer.forward(self, x)` must return `self.fc2(self.fc1(x))` (no activation — this drill is about REGISTRATION, not nonlinearity).

Because both `fc1` and `fc2` are assigned as attributes inside `__init__`, the base `nn.Module.__setattr__` should auto-register them, and `list(model.parameters())` should yield 4 tensors (2 weights + 2 biases) in registration order.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

class TwoLayer(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, out_dim: int):
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError


def cx19_build_two_layer(in_dim: int, hid_dim: int, out_dim: int) -> 'TwoLayer':
    raise NotImplementedError

def _test_cx19():
    # Case A: instance is an nn.Module and has the right submodules registered.
    model = cx19_build_two_layer(4, 8, 3)
    assert isinstance(model, nn.Module), 'TwoLayer must subclass nn.Module'
    assert isinstance(model, TwoLayer), 'builder must return a TwoLayer instance'
    assert hasattr(model, 'fc1') and isinstance(model.fc1, nn.Linear), 'fc1 must be nn.Linear'
    assert hasattr(model, 'fc2') and isinstance(model.fc2, nn.Linear), 'fc2 must be nn.Linear'
    assert model.fc1.in_features == 4 and model.fc1.out_features == 8
    assert model.fc2.in_features == 8 and model.fc2.out_features == 3

    # Case B: AUTO-registration via __setattr__ — both children appear in _modules.
    child_names = list(model._modules.keys())
    assert child_names == ['fc1', 'fc2'], f'child registration order broken: {child_names}'

    # Case C: .parameters() recurses into BOTH children — 4 tensors total.
    ps = list(model.parameters())
    assert len(ps) == 4, f'expected 4 params (2 W + 2 b), got {len(ps)}'
    named = dict(model.named_parameters())
    assert set(named.keys()) == {'fc1.weight', 'fc1.bias', 'fc2.weight', 'fc2.bias'}, named.keys()

    # Case D: forward composes fc2(fc1(x)) — shape and value match the manual chain.
    t.manual_seed(0)
    model = cx19_build_two_layer(5, 7, 2)
    x = t.randn(3, 5)
    y = model(x)
    assert tuple(y.shape) == (3, 2), f'expected (3, 2), got {tuple(y.shape)}'
    y_ref = model.fc2(model.fc1(x))
    assert t.allclose(y, y_ref, atol=1e-6), 'forward must be fc2(fc1(x)), no activation'

    # Case E: .to(device) and .train() reach the children — proves registration worked.
    model.eval()
    assert not model.fc1.training and not model.fc2.training, 'eval() must propagate to children'
    model.train()
    assert model.fc1.training and model.fc2.training, 'train() must propagate to children'
    _dd_passed.add('cx19')

_test_cx19()

<details><summary>Show solution — cx19</summary>

```python
class TwoLayer(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, out_dim: int):
        # Atom A (nn-module-subclass): super().__init__() FIRST so _modules exists
        # before we start assigning child modules.
        super().__init__()
        # Atom B (module-composition): assigning nn.Module instances to attrs triggers
        # nn.Module.__setattr__, which registers them in self._modules under the attr name.
        self.fc1 = nn.Linear(in_dim, hid_dim)
        self.fc2 = nn.Linear(hid_dim, out_dim)

    def forward(self, x):
        # Children are called as __call__ — that runs hooks + forward.
        return self.fc2(self.fc1(x))


def cx19_build_two_layer(in_dim: int, hid_dim: int, out_dim: int) -> 'TwoLayer':
    return TwoLayer(in_dim, hid_dim, out_dim)
```

**Order of operations in `__init__` is load-bearing.** If you assign `self.fc1` BEFORE `super().__init__()`, the auto-registration silently no-ops (no `_modules` dict yet) and `.parameters()` returns an empty iterator — a classic ARENA bug because forward still works.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx19'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx19',
        'subtopics': ["PyTorch: nn.Module subclassing", "PyTorch: Module composition"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()